# Veyr training-data exploration

Exploratory analysis of `~/.veyr/ml/training-data.jsonl` — the per-turn samples
the Veyr Mac app collects (labeled via Agent-tab session ratings).

Dev-side only; see `README.md`. Run `pip install -r requirements.txt` first.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

from train import LABELS, Sample, heuristic_predict, load_labeled_samples, load_current_heuristics

DATA = Path.home() / ".veyr" / "ml" / "training-data.jsonl"
CONFIG = Path.home() / ".veyr" / "config.json"

rows = []
for line in DATA.read_text().splitlines():
    line = line.strip()
    if not line:
        continue
    try:
        rows.append(json.loads(line))
    except json.JSONDecodeError:
        pass
df = pd.DataFrame(rows)
print(f"{len(df)} samples, {df['userFeedbackComplexity'].notna().sum()} labeled")
df.head()

## Class balance

How the Haiku classifier labels turns vs. your session ratings.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
df["llmClassification"].value_counts().reindex(LABELS).plot.bar(ax=axes[0], title="Haiku classification", color="#4FABFF")
labeled = df[df["userFeedbackComplexity"].notna()]
labeled["userFeedbackComplexity"].value_counts().reindex(LABELS).plot.bar(ax=axes[1], title="Your ratings (ground truth)", color="#34d399")
plt.tight_layout()

## Message length by label

The shipped heuristic relies heavily on length — does the labeled data support that?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for label, color in zip(LABELS, ["#34d399", "#f5a623", "#4FABFF"]):
    subset = labeled.loc[labeled["userFeedbackComplexity"] == label, "userMessageLength"]
    if len(subset):
        subset.plot.hist(ax=ax, bins=40, alpha=0.5, label=f"{label} (n={len(subset)})", color=color)
ax.set_xlabel("user message length (chars)")
ax.legend()
ax.set_title("Message length distribution per ground-truth label")

## Heuristic confusion matrix

Where `quickComplexityEstimate` (current thresholds) disagrees with your labels.

In [ ]:
samples = load_labeled_samples(DATA)
simple_max, complex_min = load_current_heuristics(CONFIG)
print(f"current thresholds: simple<{simple_max}, complex>{complex_min}")

confusion = pd.DataFrame(0, index=LABELS, columns=LABELS)
for s in samples:
    confusion.loc[s.label, heuristic_predict(s, simple_max, complex_min)] += 1
confusion.index.name = "truth ↓ / predicted →"
confusion

## Feature importances

Which collected features actually separate the classes.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

from train import FEATURE_NAMES, feature_matrix

X = feature_matrix(samples)
y = [s.label for s in samples]
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X, y)
pd.Series(tree.feature_importances_, index=FEATURE_NAMES).sort_values().plot.barh(
    figsize=(7, 3), color="#4FABFF", title="Decision-tree feature importance")

## Next steps

- `python train.py --dry-run` to see whether upgraded thresholds beat the shipped ones
- When the full-feature model consistently beats length-only thresholds, wire
  model inference into the runtimes (export coefficients to JSON, or Core ML
  for the Swift side)